# Ebaluazioa 3.1.3: Klasifikazio Probabilistikoa (MNIST) - Ensemble Eredua
Helburua Random Forest eta Sare Neuronal (NN_MCDropout) baten probabilitateak bateratzea da, bakoitzaren ahuleziak estaltzeko. 3 zenbakia itzalean utziko dugu probetarako.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.generators import MNISTGenerator
from src.noise import AbsoluteGaussianNoise
from src.shadows import ClassShadow
from src.classifiers import NeuralNetworkMCDropoutClassifier, RandomForestClassifierModel, BayesianNeuralNetworkClassifier
from src.ensemble import EnsembleClassifier
from src.splitter import GapSplitter
from src.orchestrator import ClassificationOrchestrator
from src.visualization import plot_mnist_classification_results, plot_mnist_sample_uncertainties

In [ ]:
N_SAMPLES = 5000 
HIDDEN_CLASS = 3
EPOCHS = 20

generator = MNISTGenerator(train=True)
shadow = ClassShadow(shadow_class=HIDDEN_CLASS)
splitter = GapSplitter(test_size=0.2, random_state=42)
noise_strategy = AbsoluteGaussianNoise(std_dev=0.0)

In [ ]:
rfc = RandomForestClassifierModel(n_estimators=50, max_depth=15, random_state=42)
nn = NeuralNetworkMCDropoutClassifier(hidden_dim=128, num_classes=10, dropout_rate=0.3, epochs=EPOCHS, n_iter=30)
models = [
    rfc,
    nn,
    EnsembleClassifier([rfc, nn])
]

In [ ]:
print(f"\n{'='*60}")
print(f"EBALUAZIOA: MNIST datu-basea ({HIDDEN_CLASS} Klasea Ezkutatuz)")
print(f"{'='*60}")

for m in models:
    print(f"\n-> Modeloa entropiarekin: {m.get_name()}")

    orchestrator = ClassificationOrchestrator(generator, splitter, m)
    artifacts = orchestrator.run(
        n_samples=N_SAMPLES,
        x_range=(0,0), 
        noise_strategy=noise_strategy,
        shadow=shadow
    )

    if artifacts.uncertainties is not None:
        ent_vis = np.mean(artifacts.uncertainties['visible'])
        ent_sha = np.mean(artifacts.uncertainties['shadow'])
        print(f"    * Zehaztasuna (Ikusgai): {artifacts.metrics.acc_test_visible:.4f}")
        print(f"    * Zehaztasuna (Itzala) : {artifacts.metrics.acc_test_shadow:.4f}")
        print(f"    * Entropia (Ikusgai)   : {ent_vis:.4f}")
        print(f"    * Entropia (Itzala)    : {ent_sha:.4f}")

    plot_mnist_classification_results(artifacts.split_data, artifacts.predictions, artifacts.uncertainties, title=f"Eredua: {m.get_name()}")
    plot_mnist_sample_uncertainties(artifacts.split_data, artifacts.predictions, artifacts.uncertainties, num_samples=5)
    
print("\nEsperimentua Arrakastaz Bideratua")

## OOD Uncertainty Interpreter - Balioespen Konbinatu Sakona
Bultzatutako interpretatzailearen bidez, entropia erabiliz 'Ezagun' eta 'Ezezagun' klaseen arteko bereizketa egingo dugu hainbat ezkutuko-konbinaketatarako. Honekin ikusiko dugu hiru ereduen artetik (RFC, NN eta Ensemble) zeinek ematen duen informazio aberatsena banaketa egiteko.

In [ ]:
import pandas as pd
from src.interpreter import UncertaintyInterpreter

combinations = [[1], [3], [8], [0, 8], [3, 5, 8], [0, 6, 9], [1, 7], [2, 4, 6], [1, 3, 5, 7, 9], [0, 1, 2, 3, 4]]
results = []

for hid_classes in combinations:
    print(f"\n{'='*60}\nEbaluatzen: Itzalean {hid_classes}\n{'='*60}")
    shadow_multi = ClassShadow(shadow_class=hid_classes)

    rfc_multi = RandomForestClassifierModel(n_estimators=50, max_depth=15, random_state=42)
    nn_multi = NeuralNetworkMCDropoutClassifier(hidden_dim=128, num_classes=10, dropout_rate=0.3, epochs=EPOCHS, n_iter=30)
    ens_multi = EnsembleClassifier([rfc_multi, nn_multi])

    for m in [rfc_multi, nn_multi, ens_multi]:
        orch = ClassificationOrchestrator(generator, splitter, m)
        arts = orch.run(n_samples=N_SAMPLES, x_range=(0,0), noise_strategy=noise_strategy, shadow=shadow_multi)

        if arts.uncertainties is not None:
            ent_vis = arts.uncertainties['visible']
            ent_sha = arts.uncertainties['shadow']

            interpreter = UncertaintyInterpreter()

            eval_res = interpreter.evaluate(ent_vis, ent_sha)

            results.append({
                'Hidden': str(hid_classes),
                'Model': m.get_name(),
                'Acc (Visible)': arts.metrics.acc_test_visible,
                'Interpreter Acc': eval_res['accuracy'],
                'Threshold': eval_res['threshold']
            })
            print(f"  -> {m.get_name():<50}: Interpreter Accuracy = {eval_res['accuracy']:.4f}")

df_results = pd.DataFrame(results)
print("\n\n" + "*"*80)
print("                     AMAIERAKO EMAITZAK                     ")
print("*"*80)
print(df_results.to_string(index=False))
